In [ ]:
import os
os.environ['HF_TOKEN'] = "YOUR_HF_TOKEN_HERE"
os.environ['NVIDIA_API_KEY'] = "YOUR_NVIDIA_API_KEY_HERE"

API_URL = "https://integrate.api.nvidia.com/v1/chat/completions"
MODEL_NAME = "stepfun-ai/step-3.5-flash"
DRIVE_OUTPUT_DIR = '/content/drive/MyDrive/ClinicSpring2026_Results'
OUTPUT_PATH = os.path.join(DRIVE_OUTPUT_DIR, "benchmark_results.json")

MAX_WORKERS = 5 # Reduced from 10 to avoid 429 rate limit errors
REQUEST_TIMEOUT = 120
API_KEY = os.getenv("NVIDIA_API_KEY")


In [ ]:
!pip install -q datasets requests tqdm


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
import json, time, re, requests, os
from enum import Enum
from typing import List, Dict, Any
from concurrent.futures import ThreadPoolExecutor, as_completed
from datasets import load_dataset
from tqdm.notebook import tqdm

class ReasoningMode(Enum):
    ZERO_SHOT = "zero_shot"
    CHAIN = "chain_of_thought"
    TREE = "tree_of_thought"
    VERIFY = "self_verification"
    FEW_SHOT = "few_shot"

def get_prompt(prompt, mode):
    format_instr = (
        "Keep your reasoning concise and focused. Avoid unnecessary repetition. "
        "Clearly state the final arrangement at the end using ONLY this format:\n"
        "Room 1: [Person], [Pet]\n"
        "Room 2: [Person], [Pet]\n"
        "..."
    )

    legend = (
        "Symbol Legend:\n"
        "- 'X @ Y' means X and Y are in the same room.\n"
        "- 'X <> Y' means X and Y are in DIFFERENT rooms.\n"
        "- 'X: o o # o' refers to the room matching the '#' position (e.g., Room 3).\n"
        "- 'X: o x o o' means X is NOT in the room matching the 'x' position (e.g., Room 2).\n"
        "- '=' and '!=' often refer to adjacency or specific exclusion constraints.\n\n"
    )

    if mode == ReasoningMode.ZERO_SHOT:
        return f"{legend}Question:\n{prompt}\n\nPlease quickly, just give me the answer to this logic puzzle without any explanation. {format_instr}"
    elif mode == ReasoningMode.CHAIN:
        return f"{legend}Question:\n{prompt}\n\nLet's think step by step:\n1) List constraints.\n2) Deduce implications.\n3) Explore possibilities.\n4) Construct final solution.\n\n{format_instr}"
    elif mode == ReasoningMode.TREE:
        return f"{legend}Question:\n{prompt}\n\nSolve using a Tree-of-Thought approach:\n1) Tree Search & Branching\n2) Candidate Evaluation\n3) Branch Selection\n4) Final Conclusion.\n\nConsensus Solution:\n{format_instr}"
    elif mode == ReasoningMode.VERIFY:
        return f"{legend}Question:\n{prompt}\n\nFirst, solve the puzzle. Then, verify against every constraint.\nFinal Verified Solution:\n{format_instr}"
    elif mode == ReasoningMode.FEW_SHOT:
        example = "Example:\nAlice, Bob, Charlie with pets: anole, bat, cat.\n1. Bob @ anole\nSolution:\nRoom 1: Bob, anole\nRoom 2: Alice, cat\nRoom 3: Charlie, bat\n\n"
        return f"{legend}{example}Question:\n{prompt}\n\n{format_instr}"
    return prompt

# --- Robust Grading Engine ---
def extract_entities(prompt):
    p_clean = prompt.split("Restrictions")[0].split("1 .")[0]
    m_ppl = re.search(r"(?:assigned to|tenants:)\s+([^.]+?)(?:\s+with pets:|\s+and pets:|\.)", p_clean, re.IGNORECASE)
    m_pts = re.search(r"pets:\s+([^.]+?)(?:\.|\bto \d+ rooms|You should)", p_clean, re.IGNORECASE)
    _SPLIT = re.compile(r",|\s+and\s+")
    def clean(s): return {x.strip().lower() for x in _SPLIT.split(s) if x.strip() and len(x.strip()) > 1}
    people = clean(m_ppl.group(1)) if m_ppl else {w for w in re.findall(r"\b[A-Z][a-z]+\b", p_clean)}
    pets = clean(m_pts.group(1)) if m_pts else set()
    return people, pets

def parse_assignments(text, people_set, pets_set):
    assignments = {}
    text = re.sub(r"<think>.*?</think>|--- Reasoning ---.*?\n\n", "", str(text), flags=re.IGNORECASE|re.DOTALL)
    matches = re.finditer(r"(?:Room|#|\b)\s*(\d+)\s*[:.-]\s*(.*?)(?=(?:Room|#|\b)\s*\d+\s*[:.-]|$)", text, re.IGNORECASE | re.DOTALL)
    for m in matches:
        rid = m.group(1)
        words = {w.lower() for w in re.findall(r"\b\w+\b", m.group(2))}
        found = {w for w in words if w in people_set or w in pets_set}
        if found: assignments[rid] = found
    if not assignments:
        for i, line in enumerate(text.splitlines()):
            words = {w.lower() for w in re.findall(r"\b\w+\b", line)}
            found = {w for w in words if w in people_set or w in pets_set}
            if found: assignments[f"idx_{i}"] = found
    return assignments

def grade(pred, gt, prompt):
    if not pred or not gt: return 0.0
    ppl, pts = extract_entities(prompt)
    if not ppl: return 0.0
    gt_map = parse_assignments(gt, ppl, pts)
    res_map = parse_assignments(pred, ppl, pts)
    earned, total = 0, (len(gt_map) * 2)
    for rid, gt_occ in gt_map.items():
        res_occ = res_map.get(rid, set())
        if any(p in res_occ for p in gt_occ if p in ppl): earned += 1
        if any(t in res_occ for t in gt_occ if t in pts): earned += 1
    def get_pairs(m):
        return {frozenset([w for w in occ if w in ppl or w in pts]) for occ in m.values() if len(occ) >= 2}
    gt_pairs, res_pairs = get_pairs(gt_map), get_pairs(res_map)
    earned += len(gt_pairs & res_pairs)
    total += len(gt_pairs)
    return earned / total if total > 0 else 0.0

def call_api(prompt, max_retries=3):
    headers = {"Authorization": f"Bearer {API_KEY}", "Content-Type": "application/json"}
    payload = {
        "model": MODEL_NAME,
        "messages": [
            {"role": "system", "content": "detailed thinking off"},
            {"role": "user", "content": prompt}
        ],
        "temperature": 0.3,
        "max_tokens": 1024,
        "enable_thinking": False,
        "thinking_budget": 0
    }
    for attempt in range(max_retries):
        try:
            r = requests.post(API_URL, json=payload, headers=headers, timeout=REQUEST_TIMEOUT)
            if r.status_code == 429: time.sleep((attempt + 1) * 3); continue
            if r.status_code != 200: return {"error": f"HTTP {r.status_code}: {r.text}"}
            msg = r.json()["choices"][0]["message"]
            content = msg.get("content") or ""
            reasoning = msg.get("reasoning_content") or msg.get("reasoning") or ""
            full = f"--- Reasoning ---\n{reasoning}\n\n{content}" if reasoning else content
            return {"response": full, "latency": r.elapsed.total_seconds()}
        except Exception as e:
            if attempt == max_retries - 1: return {"error": str(e)}
            time.sleep(2)
    return {"error": "Max retries exceeded"}

def save_results(results, path=None):
    p = path or OUTPUT_PATH
    os.makedirs(os.path.dirname(p), exist_ok=True)
    with open(p, "w") as f: json.dump(results, f, indent=2)


In [ ]:
def run(limit=1000):
    print("Loading dataset...")
    ds = load_dataset("emunah/deductive_logical_reasoning-room_assignment", split="train", token=os.getenv("HF_TOKEN"))
    ans_k = "completion" if "completion" in ds.column_names else "answer"
    ds = ds.shuffle(seed=42).select(range(min(limit, len(ds))))
    all_summary = []

    for mode in ReasoningMode:
        print(f"\n=== Testing Method: {mode.name} ===")
        results = []
        out = os.path.join(DRIVE_OUTPUT_DIR, f"results_{mode.name.lower()}.json")
        
        def process(i):
            item = ds[i]
            res = call_api(get_prompt(item["question"], mode))
            if "error" in res: return {"error": res["error"]}
            return {
                "score": grade(res["response"], item[ans_k], item["question"]),
                "latency": res["latency"],
                "mode": mode.name
            }

        with ThreadPoolExecutor(max_workers=MAX_WORKERS) as ex:
            futures = [ex.submit(process, i) for i in range(len(ds))]
            for f in tqdm(as_completed(futures), total=len(futures), desc=mode.name):
                results.append(f.result())
        
        save_results(results, out)
        scores = [r["score"] for r in results if "error" not in r]
        avg = sum(scores)/len(scores) if scores else 0
        print(f"Mode {mode.name} Average Score: {avg:.4f}")
        all_summary.append((mode.name, avg))

    print("\n" + "="*30 + "\nFINAL SUMMARY\n" + "="*30)
    for name, score in all_summary: print(f"{name:15}: {score:.4f}")

run(limit=1000)
